# Gold Transformation Pipeline

Lakeflow Declarative Pipeline stage that builds analytical tables in the **serve** schema from cleaned **refined** tables.

Produces nine gold tables: four dimensional summaries plus five dashboard-ready datasets (headline KPIs, sales/purchase trends, top selling materials).

**Prerequisite:** `ldp_silver_transformations` must complete successfully first.

**Workflow usage:** run as the second pipeline task, depending on the silver stage.


## Configuration


In [ ]:
CATALOG = "jm_databricks_learning_ws"
REFINED_SCHEMA = "refined"
SERVE_SCHEMA = "serve"

REFINED = f"{CATALOG}.{REFINED_SCHEMA}"
SERVE = f"{CATALOG}.{SERVE_SCHEMA}"


## Imports


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    add_months,
    coalesce,
    col,
    count,
    countDistinct,
    date_format,
    lit,
    max as spark_max,
    row_number,
    sum as spark_sum,
    trunc,
)
from pyspark.sql.window import Window

TOP_SELLING_MATERIALS_LIMIT = 10
TREND_MONTHS = 12


## `serve.supplier_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.supplier_summary",
    comment="Purchase order volume and quantity by supplier",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_supplier_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    suppliers = spark.read.table(f"{REFINED}.suppliers")

    return (
        purchase_orders.join(suppliers, on="supplier_id", how="inner")
        .groupBy(
            col("supplier_id"),
            col("supplier_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_purchase_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.customer_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.customer_summary",
    comment="Sales order volume and quantity by customer",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_customer_summary():
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    customers = spark.read.table(f"{REFINED}.customers")

    return (
        sales_orders.join(customers, on="customer_id", how="inner")
        .groupBy(
            col("customer_id"),
            col("customer_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_sales_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.inventory_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.inventory_summary",
    comment="Current stock and material coverage by warehouse",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_inventory_summary():
    inventory = spark.read.table(f"{REFINED}.inventory")

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = inventory.join(
        latest_snapshot,
        inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
        how="inner",
    ).drop("latest_snapshot_date")

    warehouses = spark.read.table(f"{REFINED}.warehouses")

    return (
        current_inventory.join(warehouses, on="warehouse_id", how="inner")
        .groupBy(
            col("warehouse_id"),
            col("warehouse_name"),
            col("plant_id"),
        )
        .agg(
            spark_sum("quantity").alias("current_stock"),
            countDistinct("material_id").alias("total_materials"),
        )
    )


## `serve.material_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.material_summary",
    comment="Purchased, sold, and on-hand quantities by material",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_material_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    inventory = spark.read.table(f"{REFINED}.inventory")
    materials = spark.read.table(f"{REFINED}.materials")

    purchased = purchase_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("purchased_quantity"),
    )

    sold = sales_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("sold_quantity"),
    )

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = (
        inventory.join(
            latest_snapshot,
            inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
            how="inner",
        )
        .groupBy("material_id")
        .agg(spark_sum("quantity").alias("current_inventory"))
    )

    return (
        materials.select("material_id", "material_name", "material_type")
        .join(purchased, on="material_id", how="left")
        .join(sold, on="material_id", how="left")
        .join(current_inventory, on="material_id", how="left")
        .select(
            "material_id",
            "material_name",
            "material_type",
            coalesce(col("purchased_quantity"), lit(0)).alias("purchased_quantity"),
            coalesce(col("sold_quantity"), lit(0)).alias("sold_quantity"),
            coalesce(col("current_inventory"), lit(0)).alias("current_inventory"),
        )
    )


## `serve.business_kpi_summary`

Single-row headline KPIs for dashboard counter widgets.

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.business_kpi_summary",
    comment="Headline business KPIs for dashboard counter widgets",
    table_properties={"quality": "gold", "domain": "executive"},
)
def serve_business_kpi_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    inventory = spark.read.table(f"{REFINED}.inventory")

    active_purchase_orders = purchase_orders.filter(col("status") != "CANCELLED")
    active_sales_orders = sales_orders.filter(col("status") != "CANCELLED")

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("inventory_snapshot_date"))
    current_inventory = inventory.join(
        latest_snapshot,
        inventory.snapshot_date == latest_snapshot.inventory_snapshot_date,
        how="inner",
    )

    purchase_stats = active_purchase_orders.agg(
        count(lit(1)).alias("total_purchase_orders"),
        countDistinct("supplier_id").alias("active_suppliers"),
    )
    sales_stats = active_sales_orders.agg(
        count(lit(1)).alias("total_sales_orders"),
        countDistinct("customer_id").alias("active_customers"),
    )
    inventory_stats = current_inventory.agg(
        coalesce(spark_sum("quantity"), lit(0)).alias("inventory_on_hand"),
    )

    return (
        purchase_stats.crossJoin(sales_stats)
        .crossJoin(inventory_stats)
        .crossJoin(latest_snapshot)
    )

## `serve.sales_trend_monthly`

Monthly sales order volume for the trailing 12 months (line chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.sales_trend_monthly",
    comment="Monthly sales order count and quantity for dashboard trend charts",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_sales_trend_monthly():
    sales_orders = (
        spark.read.table(f"{REFINED}.sales_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )

    latest_month = sales_orders.agg(
        spark_max(trunc("order_date", "month")).alias("latest_month")
    )
    month_offsets = spark.range(0, TREND_MONTHS)
    months = latest_month.crossJoin(month_offsets).select(
        add_months(
            col("latest_month"),
            col("id").cast("int") - (TREND_MONTHS - 1),
        ).alias("month_start_date")
    )

    monthly_totals = (
        sales_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("month_start_date")
        .agg(
            count(lit(1)).alias("order_count"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )

    return (
        months.join(monthly_totals, on="month_start_date", how="left")
        .select(
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("order_count"), lit(0)).alias("order_count"),
            coalesce(col("total_quantity"), lit(0)).alias("total_quantity"),
        )
        .orderBy("month_start_date")
    )

## `serve.purchase_trend_monthly`

Monthly purchase order volume for the trailing 12 months (line chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.purchase_trend_monthly",
    comment="Monthly purchase order count and quantity for dashboard trend charts",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_purchase_trend_monthly():
    purchase_orders = (
        spark.read.table(f"{REFINED}.purchase_orders")
        .filter(col("status") != "CANCELLED")
        .filter(col("order_date").isNotNull())
    )

    latest_month = purchase_orders.agg(
        spark_max(trunc("order_date", "month")).alias("latest_month")
    )
    month_offsets = spark.range(0, TREND_MONTHS)
    months = latest_month.crossJoin(month_offsets).select(
        add_months(
            col("latest_month"),
            col("id").cast("int") - (TREND_MONTHS - 1),
        ).alias("month_start_date")
    )

    monthly_totals = (
        purchase_orders.withColumn("month_start_date", trunc("order_date", "month"))
        .groupBy("month_start_date")
        .agg(
            count(lit(1)).alias("order_count"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )

    return (
        months.join(monthly_totals, on="month_start_date", how="left")
        .select(
            col("month_start_date"),
            date_format("month_start_date", "yyyy-MM").alias("year_month"),
            coalesce(col("order_count"), lit(0)).alias("order_count"),
            coalesce(col("total_quantity"), lit(0)).alias("total_quantity"),
        )
        .orderBy("month_start_date")
    )

## `serve.top_selling_materials`

Top finished goods by sold quantity (bar chart).

In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.top_selling_materials",
    comment="Top finished goods ranked by sold quantity for dashboard bar charts",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_top_selling_materials():
    sales_orders = spark.read.table(f"{REFINED}.sales_orders").filter(
        col("status") != "CANCELLED"
    )
    materials = spark.read.table(f"{REFINED}.materials").filter(
        col("material_type") == "FINISHED_GOOD"
    )

    sold_by_material = sales_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("sold_quantity"),
    )

    ranked = (
        sold_by_material.join(materials, on="material_id", how="inner")
        .withColumn(
            "sales_rank",
            row_number().over(Window.orderBy(col("sold_quantity").desc(), col("material_id"))),
        )
        .filter(col("sales_rank") <= TOP_SELLING_MATERIALS_LIMIT)
    )

    return ranked.select(
        "sales_rank",
        "material_id",
        "material_name",
        "material_type",
        "sold_quantity",
    ).orderBy("sales_rank")